In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/muhammmadsarmad/nail-disease/README.dataset.txt
/kaggle/input/datasets/muhammmadsarmad/nail-disease/README.roboflow.txt
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/ps-nails_jpg.rf.d8abd9bc29aa882dba09cb53c5cf051b.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/Nail-Psoriasis-419-_jpeg.rf.2e538b56ccee6cf81ac1859d5bd47692.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/psoriasis-3_jpg.rf.56629a3c2c8531cc6f8124e3368b6ff1.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/psoriasis-135_jpg.rf.0801949d3a13bcabaedf7e879ace0f01.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/psoriasis-137_jpg.rf.68014e3d68d3c3250b0f8940ce53e728.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/ps-nails_jpg.rf.cc31b7ddf3038b5c2de14a2d74cd7ac1.jpg
/kaggle/input/datasets/muhammmadsarmad/nail-disease/valid/Nail_Psoriasis/Nai

In [2]:
# Step 1: Import libraries needed for file handling, hashing, image processing, and duplicate detection
import os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

!pip install imagehash -q
import imagehash

In [3]:
# Step 2: Auto-detect dataset root since Kaggle input paths aren't fixed
def find_data_root():
    kaggle_input = Path("/kaggle/input")
    for p in kaggle_input.rglob("train"):
        if p.is_dir() and (p.parent / "valid").exists() and (p.parent / "test").exists():
            print(f"Found dataset at: {p.parent}")
            return p.parent
    print("[WARN] Not found. Contents of /kaggle/input:")
    for p in kaggle_input.iterdir():
        print(" -", p)
    return None

DATA_ROOT = find_data_root()

Found dataset at: /kaggle/input/datasets/muhammmadsarmad/nail-disease


In [4]:
# Step 3: Set up splits, classes, label mapping, output path, and image size config
SPLITS = ["train", "valid", "test"]
CLASSES = sorted(os.listdir(DATA_ROOT / "train"))
LABEL_MAP = {name: i for i, name in enumerate(CLASSES)}
OUTPUT_DIR = Path("/kaggle/working/preprocessed_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = (224, 224)
PHASH_THRESHOLD = 5
print(LABEL_MAP)

{'ALM': 0, 'Blue_finger': 1, 'Clubbing': 2, 'Healthy_Nail': 3, 'Nail_Psoriasis': 4, 'Onychogryphosis': 5, 'Onychomycosis': 6, 'Pitting': 7}


In [5]:
# Step 4: Build a dataframe listing every image's path, split, class, and label
records = []
for split in SPLITS:
    for cls in CLASSES:
        d = DATA_ROOT / split / cls
        for fname in sorted(os.listdir(d)):
            fpath = d / fname
            if fpath.is_file():
                records.append({"path": str(fpath), "filename": fname,
                                 "split": split, "class_name": cls, "label": LABEL_MAP[cls]})
df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
df.head()

Total images: 5886


,path,filename,split,class_name,label
0,/kaggle/input/datasets/muhammmadsarmad/nail-di...,ALM-1-_jpg.rf.add48734f8e6147ad0d63db616b15970...,train,ALM,0
1,/kaggle/input/datasets/muhammmadsarmad/nail-di...,ALM-100-_jpg.rf.34657a1fac88d79271fc1ba782bd52...,train,ALM,0
2,/kaggle/input/datasets/muhammmadsarmad/nail-di...,ALM-101-_jpg.rf.015506a69eb8561bce611cf1b2ab2e...,train,ALM,0
3,/kaggle/input/datasets/muhammmadsarmad/nail-di...,ALM-102-_jpg.rf.743439b9d039410a0bedb12fec3ece...,train,ALM,0
4,/kaggle/input/datasets/muhammmadsarmad/nail-di...,ALM-103-_jpg.rf.f70ec2a0f1f67c6f157e69df15d7c7...,train,ALM,0


In [6]:
# Step 5: Validate every image and drop corrupt/unreadable files before training
widths, heights, valid_flags = [], [], []
for path in df["path"]:
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            w, h = im.size
        widths.append(w); heights.append(h); valid_flags.append(True)
    except (UnidentifiedImageError, OSError):
        widths.append(None); heights.append(None); valid_flags.append(False)
df["width"] = widths
df["height"] = heights
df["valid"] = valid_flags
n_bad = (~df["valid"]).sum()
if n_bad:
    print(f"Dropping {n_bad} corrupt images")
df = df[df["valid"]].drop(columns=["valid"]).reset_index(drop=True)
print(f"Valid images: {len(df)}")

Valid images: 5886


In [7]:
# Step 6: Detect exact duplicate images via MD5 hash to avoid data leakage across splits
md5_hashes = []
for path in df["path"]:
    with open(path, "rb") as f:
        md5_hashes.append(hashlib.md5(f.read()).hexdigest())
df["md5"] = md5_hashes
dup_mask = df.duplicated(subset="md5", keep="first")
print(f"Exact duplicates: {dup_mask.sum()}")
cross_split_leaks = 0
for md5, group in df.groupby("md5"):
    if group["split"].nunique() > 1:
        cross_split_leaks += len(group) - 1
print(f"Cross-split exact duplicates (leakage): {cross_split_leaks}")
df["exact_duplicate"] = dup_mask

Exact duplicates: 0
Cross-split exact duplicates (leakage): 0


In [8]:
# Step 7: Detect near-duplicate images via perceptual hash since MD5 misses visually similar images
phashes = []
for path in df["path"]:
    with Image.open(path) as im:
        phashes.append(imagehash.phash(im.convert("RGB")))
df["phash"] = phashes
near_dup_flags = [False] * len(df)
cross_split_near = 0
seen = []
for i, h in enumerate(df["phash"]):
    if df.loc[i, "exact_duplicate"]:
        continue
    matched = None
    for j, h2, s2 in seen:
        if h - h2 <= PHASH_THRESHOLD:
            matched = (j, s2)
            break
    if matched is not None:
        near_dup_flags[i] = True
        if df.loc[i, "split"] != matched[1]:
            cross_split_near += 1
    else:
        seen.append((i, h, df.loc[i, "split"]))
df["near_duplicate"] = near_dup_flags
print(f"Near duplicates: {sum(near_dup_flags)}")
print(f"Cross-split near duplicates (leakage risk): {cross_split_near}")
df = df.drop(columns=["phash"])
df["drop_duplicate"] = df["exact_duplicate"] | df["near_duplicate"]

Near duplicates: 119
Cross-split near duplicates (leakage risk): 38


In [9]:
# Step 8: Check dataset size and image dimension stats after removing duplicates
clean = df[~df["drop_duplicate"]]
print(f"After dedup: {len(clean)} unique images remain")
print("Width stats:\n", clean["width"].describe()[["min", "max", "50%"]])
print("Height stats:\n", clean["height"].describe()[["min", "max", "50%"]])

After dedup: 5767 unique images remain
Width stats:
 min      68.0
max    1848.0
50%     448.0
Name: width, dtype: float64
Height stats:
 min      75.0
max    1633.0
50%     448.0
Name: height, dtype: float64


In [10]:
# Step 9: Resize and normalize images, then save as .npy arrays per split for fast loading during training
clean = df[~df["drop_duplicate"]].reset_index(drop=True)
for split in SPLITS:
    split_df = clean[clean["split"] == split]
    X = np.zeros((len(split_df), IMG_SIZE[1], IMG_SIZE[0], 3), dtype=np.float32)
    for i, (_, row) in enumerate(split_df.iterrows()):
        with Image.open(row["path"]) as im:
            im = im.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
            X[i] = np.asarray(im, dtype=np.float32) / 255.0
    y = split_df["label"].to_numpy()
    np.save(OUTPUT_DIR / f"X_{split}.npy", X)
    np.save(OUTPUT_DIR / f"y_{split}.npy", y)
    print(f"Saved {split}: X{X.shape}, y{y.shape}")

Saved train: X(4611, 224, 224, 3), y(4611,)
Saved valid: X(860, 224, 224, 3), y(860,)
Saved test: X(296, 224, 224, 3), y(296,)


In [11]:
# Step 10: Save metadata and label map to disk for reproducibility and later reference
df.to_csv(OUTPUT_DIR / "metadata.csv", index=False)
pd.DataFrame(list(LABEL_MAP.items()), columns=["class_name", "label"]).to_csv(OUTPUT_DIR / "label_map.csv", index=False)
print("Final split sizes:")
print(clean.groupby(["split", "class_name"]).size().unstack(fill_value=0))
print("Done!")

Final split sizes:
class_name  ALM  Blue_finger  Clubbing  Healthy_Nail  Nail_Psoriasis  \
split                                                                  
test         49           31        39            38              21   
train       678          490       626           641             426   
valid       132           91       118           119              79   

class_name  Onychogryphosis  Onychomycosis  Pitting  
split                                                
test                     44             41       33  
train                   642            584      524  
valid                   103            120       98  
Done!


In [12]:
# Step 11: Check train-split class imbalance and compute class weights to handle it during training
train_counts = clean[clean["split"] == "train"]["class_name"].value_counts()
print("Train class counts:\n", train_counts)
max_count = train_counts.max()
min_count = train_counts.min()
imbalance_ratio = max_count / min_count
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}x")
print(f"Largest class: {train_counts.idxmax()} ({max_count})")
print(f"Smallest class: {train_counts.idxmin()} ({min_count})")
if imbalance_ratio > 2.0:
    print("\n[WARN] Significant class imbalance detected (>2x) — consider class_weight or oversampling.")
elif imbalance_ratio > 1.3:
    print("\n[INFO] Mild imbalance — class_weight recommended but not critical.")
else:
    print("\nClasses are reasonably balanced.")

from sklearn.utils.class_weight import compute_class_weight
y_train = clean[clean["split"] == "train"]["label"].to_numpy()
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = {int(k): float(v) for k, v in zip(np.unique(y_train), class_weights)}
print("\nSuggested class_weight dict (for model.fit):")
print(class_weight_dict)

Train class counts:
 class_name
ALM                678
Onychogryphosis    642
Healthy_Nail       641
Clubbing           626
Onychomycosis      584
Pitting            524
Blue_finger        490
Nail_Psoriasis     426
Name: count, dtype: int64

Imbalance ratio (max/min): 1.59x
Largest class: ALM (678)
Smallest class: Nail_Psoriasis (426)

[INFO] Mild imbalance — class_weight recommended but not critical.

Suggested class_weight dict (for model.fit):
{0: 0.8501106194690266, 1: 1.1762755102040816, 2: 0.9207268370607029, 3: 0.8991809672386896, 4: 1.3529929577464788, 5: 0.8977803738317757, 6: 0.986943493150685, 7: 1.0999522900763359}


In [13]:
# Step 12: Show full per-class, per-split breakdown and re-check balance on the combined total
print("=== Per-Disease Count by Split (after dedup) ===")
split_breakdown = clean.groupby(["class_name", "split"]).size().unstack(fill_value=0)
split_breakdown["Total"] = split_breakdown.sum(axis=1)
print(split_breakdown)
print(f"\n=== Grand Total (all splits combined) ===")
print(f"Total images after removing duplicates: {len(clean)}")
print("\n=== Class Balance Check (combined total) ===")
total_per_class = split_breakdown["Total"].sort_values(ascending=False)
print(total_per_class)
max_count = total_per_class.max()
min_count = total_per_class.min()
imbalance_ratio = max_count / min_count
print(f"\nLargest class: {total_per_class.idxmax()} ({max_count})")
print(f"Smallest class: {total_per_class.idxmin()} ({min_count})")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}x")
if imbalance_ratio > 2.0:
    print("[WARN] Significant imbalance (>2x) — use class_weight or oversampling.")
elif imbalance_ratio > 1.3:
    print("[INFO] Mild imbalance — class_weight recommended.")
else:
    print("Classes are reasonably balanced.")

=== Per-Disease Count by Split (after dedup) ===
split            test  train  valid  Total
class_name                                
ALM                49    678    132    859
Blue_finger        31    490     91    612
Clubbing           39    626    118    783
Healthy_Nail       38    641    119    798
Nail_Psoriasis     21    426     79    526
Onychogryphosis    44    642    103    789
Onychomycosis      41    584    120    745
Pitting            33    524     98    655

=== Grand Total (all splits combined) ===
Total images after removing duplicates: 5767

=== Class Balance Check (combined total) ===
class_name
ALM                859
Healthy_Nail       798
Onychogryphosis    789
Clubbing           783
Onychomycosis      745
Pitting            655
Blue_finger        612
Nail_Psoriasis     526
Name: Total, dtype: int64

Largest class: ALM (859)
Smallest class: Nail_Psoriasis (526)
Imbalance ratio (max/min): 1.63x
[INFO] Mild imbalance — class_weight recommended.
